In [1]:
import sys
sys.path.append("/kaggle/input/library-metrics")

In [2]:
import os, time, psutil, copy
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from metric_toolkit import evaluate_all_metrics

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [3]:
FS_METHOD = "GA_SVM"

X_train = pd.read_csv("/kaggle/input/data-time-complexity/X_train.csv")
X_val = pd.read_csv("/kaggle/input/data-time-complexity/X_val.csv")
X_test  = pd.read_csv("/kaggle/input/data-time-complexity/X_test.csv")
y_train = pd.read_csv("/kaggle/input/data-time-complexity/y_train.csv").values.ravel()
y_val = pd.read_csv("/kaggle/input/data-time-complexity/y_val.csv").values.ravel()
y_test  = pd.read_csv("/kaggle/input/data-time-complexity/y_test.csv").values.ravel()

mask = np.load("/kaggle/input/fs-result/ga_mask.npy").astype(bool)
X_train_sel = X_train.iloc[:, mask]
X_val_sel = X_val.iloc[:, mask]
X_test_sel  = X_test.iloc[:, mask]

X_train_t = torch.tensor(X_train_sel.values, dtype=torch.float32)
X_val_t = torch.tensor(X_val_sel.values, dtype=torch.float32)
X_test_t  = torch.tensor(X_test_sel.values, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
y_val_t = torch.tensor(y_val, dtype=torch.long)
y_test_t  = torch.tensor(y_test, dtype=torch.long)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=64, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=64, shuffle=True)
test_loader  = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=128, shuffle=False)

input_dim = X_train_sel.shape[1]
num_classes = len(np.unique(y_train))
print(f"Input dim: {input_dim}, Classes: {num_classes}")

Input dim: 520, Classes: 8


In [4]:
fs_metrics = pd.read_csv("/kaggle/input/fs-result/fs_metrics_ga.csv").iloc[0]
wall_time = fs_metrics["WallTime(s)"]
cpu_util  = fs_metrics["CPUUtil(%)"]
peak_mem  = fs_metrics["PeakMem(MB)"]
base_mem  = fs_metrics["BaseMem(MB)"]
energy    = fs_metrics["Energy(J)"]
carbon    = fs_metrics["Carbon(gCO2e)"]
edp       = fs_metrics["EDP(J*s)"]
cache_hit = fs_metrics.get("CacheHit", np.nan)

In [5]:
class MLP(nn.Module):
    def __init__(self, in_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, num_classes)
        )
    def forward(self, x): return self.net(x)

class LSTMModel(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_classes):
        super().__init__()
        self.lstm = nn.LSTM(in_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_classes)
    def forward(self, x):
        x = x.unsqueeze(1)
        _, (h, _) = self.lstm(x)
        return self.fc(h[-1])

class GRUModel(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_classes):
        super().__init__()
        self.gru = nn.GRU(in_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_classes)
    def forward(self, x):
        x = x.unsqueeze(1)
        _, h = self.gru(x)
        return self.fc(h[-1])

class CNN1D(nn.Module):
    def __init__(self, in_dim, num_classes):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv1d(32, 64, 3, padding=1)
        self.fc = nn.Linear(64 * in_dim, num_classes)
    def forward(self, x):
        x = x.unsqueeze(1)
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = x.flatten(1)
        return self.fc(x)

class TransformerModel(nn.Module):
    def __init__(self, in_dim, num_classes, nhead=4):
        super().__init__()
        self.embed = nn.Linear(in_dim, 128)
        enc_layer = nn.TransformerEncoderLayer(d_model=128, nhead=nhead, batch_first=True)
        self.enc = nn.TransformerEncoder(enc_layer, num_layers=2)
        self.fc = nn.Linear(128, num_classes)
    def forward(self, x):
        x = self.embed(x).unsqueeze(1)
        x = self.enc(x)
        return self.fc(x[:, 0])

class VAEModel(nn.Module):
    def __init__(self, in_dim, latent_dim, num_classes):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(in_dim, 128), nn.ReLU(), nn.Linear(128, latent_dim*2))
        self.decoder = nn.Sequential(nn.Linear(latent_dim, 128), nn.ReLU(), nn.Linear(128, num_classes))
    def reparam(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    def forward(self, x):
        h = self.encoder(x)
        mu, logvar = h.chunk(2, dim=-1)
        z = self.reparam(mu, logvar)
        return self.decoder(z)


In [6]:
def train_and_log(model_name, model, train_loader, val_loader, epochs=15, lr=1e-3, log_dir="/kaggle/working/logs"):
    os.makedirs(log_dir, exist_ok=True)
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    log = []
    best_acc = 0
    best_wts = copy.deepcopy(model.state_dict())

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0
        total_batches = 0

        # --- Training loop ---
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out = model(Xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            total_batches += 1

        train_loss = running_loss / total_batches

        # --- Validation ---
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for Xv, yv in val_loader:
                Xv, yv = Xv.to(device), yv.to(device)
                pred = model(Xv).argmax(1)
                correct += (pred == yv).sum().item()
                total += yv.size(0)
        val_acc = correct / total if total > 0 else 0

        # --- Save best weights ---
        if val_acc > best_acc:
            best_acc = val_acc
            best_wts = copy.deepcopy(model.state_dict())

        log.append({"Epoch": epoch, "TrainLoss": train_loss, "ValAcc": val_acc})
        print(f"[{model_name}] Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | Val Acc: {val_acc:.4f}")

    # --- Save log CSV ---
    log_path = os.path.join(log_dir, f"{model_name}_training_log.csv")
    pd.DataFrame(log).to_csv(log_path, index=False)
    print(f"✅ Saved training log: {log_path}")

    # --- Load best weights before returning ---
    model.load_state_dict(best_wts)
    return model, best_acc

In [7]:
# ============================================================
# 🧩 CONDITIONAL GAN TRAINING WITH LOGGING
# ============================================================

class Generator(nn.Module):
    def __init__(self, latent_dim, num_classes, output_dim):
        super().__init__()
        self.label_emb = nn.Embedding(num_classes, num_classes)
        self.net = nn.Sequential(
            nn.Linear(latent_dim + num_classes, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, output_dim),
            nn.Tanh()
        )
    def forward(self, z, labels):
        c = self.label_emb(labels)
        x = torch.cat([z, c], dim=1)
        return self.net(x)

class Discriminator(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256), nn.LeakyReLU(0.2),
            nn.Linear(256, 128), nn.LeakyReLU(0.2)
        )
        self.adv = nn.Linear(128, 1)
        self.cls = nn.Linear(128, num_classes)
    def forward(self, x):
        h = self.net(x)
        return torch.sigmoid(self.adv(h)), self.cls(h)

def train_gan_with_log(X_train, y_train, X_val, y_val, input_dim, num_classes,
                       latent_dim=64, epochs=80, lr=1e-3, batch_size=64, log_dir="/kaggle/working/logs"):
    os.makedirs(log_dir, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_loader = DataLoader(TensorDataset(torch.tensor(X_train.values, dtype=torch.float32),
                                            torch.tensor(y_train, dtype=torch.long)),
                              batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(TensorDataset(torch.tensor(X_val.values, dtype=torch.float32),
                                          torch.tensor(y_val, dtype=torch.long)),
                            batch_size=batch_size, shuffle=False)

    G = Generator(latent_dim, num_classes, input_dim).to(device)
    D = Discriminator(input_dim, num_classes).to(device)
    opt_G = optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.999))
    opt_D = optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.999))
    adv_loss, aux_loss = nn.BCELoss(), nn.CrossEntropyLoss()

    best_acc = 0
    best_D_wts = copy.deepcopy(D.state_dict())
    history = []

    for epoch in range(1, epochs + 1):
        G.train(); D.train()
        d_loss_epoch, g_loss_epoch = 0, 0
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            bs = Xb.size(0)
            valid, fake = torch.ones(bs, 1).to(device), torch.zeros(bs, 1).to(device)
            z, y_gen = torch.randn(bs, latent_dim).to(device), torch.randint(0, num_classes, (bs,)).to(device)
            gen_data = G(z, y_gen)

            # --- Train Discriminator ---
            opt_D.zero_grad()
            vr, cr = D(Xb); vf, cf = D(gen_data.detach())
            d_loss_real = adv_loss(vr, valid) + aux_loss(cr, yb)
            d_loss_fake = adv_loss(vf, fake) + aux_loss(cf, y_gen)
            d_loss = (d_loss_real + d_loss_fake) / 2
            d_loss.backward(); opt_D.step()

            # --- Train Generator ---
            opt_G.zero_grad()
            v_fake, c_fake = D(gen_data)
            g_loss = adv_loss(v_fake, valid) + aux_loss(c_fake, y_gen)
            g_loss.backward(); opt_G.step()

            d_loss_epoch += d_loss.item()
            g_loss_epoch += g_loss.item()

        # --- Validation accuracy ---
        D.eval(); correct, total = 0, 0
        with torch.no_grad():
            for Xv, yv in val_loader:
                Xv, yv = Xv.to(device), yv.to(device)
                _, out = D(Xv)
                preds = out.argmax(1)
                correct += (preds == yv).sum().item()
                total += yv.size(0)
        val_acc = correct / total

        if val_acc > best_acc:
            best_acc = val_acc
            best_D_wts = copy.deepcopy(D.state_dict())

        d_loss_epoch /= len(train_loader)
        g_loss_epoch /= len(train_loader)
        history.append({"Epoch": epoch, "D_Loss": d_loss_epoch, "G_Loss": g_loss_epoch, "ValAcc": val_acc})
        print(f"[GAN] Epoch {epoch:03d} | D_Loss={d_loss_epoch:.4f} | G_Loss={g_loss_epoch:.4f} | ValAcc={val_acc:.4f}")

    # --- Save log ---
    log_path = os.path.join(log_dir, "GAN_training_log.csv")
    pd.DataFrame(history).to_csv(log_path, index=False)
    print(f"✅ GAN training log saved: {log_path}")

    D.load_state_dict(best_D_wts)
    return D, best_acc


In [8]:
# ============================================================
# 🚀 TRAIN ALL DEEP LEARNING MODELS & SAVE RESULTS
# ============================================================

results = []
os.makedirs("/kaggle/working/logs", exist_ok=True)

# --- 1️⃣ Classical deep models ---
models = {
    "MLP": MLP(input_dim, num_classes),
    "LSTM": LSTMModel(input_dim, 64, num_classes),
    "GRU": GRUModel(input_dim, 64, num_classes),
    "CNN1D": CNN1D(input_dim, num_classes),
    "Transformer": TransformerModel(input_dim, num_classes),
    "VAE": VAEModel(input_dim, 32, num_classes)
}

for name, model in models.items():
    print(f"\n🔹 Training {name} ...")
    start = time.time()
    model, best_val_acc = train_and_log(
        model_name=name,
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=50,
        lr=1e-3,
        log_dir="/kaggle/working/logs"
    )

    # --- Evaluate on test set ---
    model.eval()
    y_true, y_pred = [], []
    y_prob = []
    with torch.no_grad():
        for Xb, yb in test_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            preds = model(Xb).argmax(1).cpu()
            prob = torch.softmax(model(Xb), dim=1).cpu().numpy()
            y_true.extend(yb.cpu().numpy())
            y_pred.extend(preds.numpy())
            y_prob.extend(prob)
    y_prob = np.array(y_prob)
    duration = time.time() - start

    metrics = evaluate_all_metrics(
        y_true=np.array(y_true),
        y_pred=np.array(y_pred),
        y_prob=y_prob,
        cpu_util=cpu_util, wall_time=wall_time,
        peak_mem_mb=peak_mem, base_mem_mb=base_mem,
        num_evals=X_train_sel.shape[1],
        missing_flags=None, labels_for_leak=y_test,
        cache_hit=cache_hit,
    )

    metrics.update({
        "FS_Method": FS_METHOD,
        "Classifier": name,
        "NumFeatures": X_train_sel.shape[1],
        "Energy(J)": energy, "Carbon(gCO2e)": carbon,
        "EDP(J*s)": edp, 
        "Timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
    })
    results.append(metrics)
    print(f"{name}: Accuracy={metrics['Accuracy']:.4f}, MacroF1={metrics['MacroF1']:.4f}")

# --- 2️⃣ GAN ---
print("\n🔹 Training GAN ...")
start = time.time()
D_gan, best_acc_gan = train_gan_with_log(
    X_train_sel, y_train, X_val_sel, y_val,
    input_dim=input_dim, num_classes=num_classes,
    latent_dim=64, epochs=50, lr=1e-3, batch_size=64,
    log_dir="/kaggle/working/logs"
)
duration = time.time() - start

# --- Evaluate GAN ---
D_gan.eval()
y_true, y_pred = [], []
y_prob = []
with torch.no_grad():
    for Xb, yb in test_loader:
        Xb, yb = Xb.to(device), yb.to(device)
        _, out = D_gan(Xb)
        prob = torch.softmax(model(Xb), dim=1).cpu().numpy()
        preds = out.argmax(1).cpu()
        y_true.extend(yb.cpu().numpy())
        y_pred.extend(preds.numpy())
        y_prob.extend(prob)
    y_prob = np.array(y_prob)
    
metrics = evaluate_all_metrics(
        y_true=np.array(y_true),
        y_pred=np.array(y_pred),
        y_prob=y_prob,
        cpu_util=cpu_util, wall_time=wall_time,
        peak_mem_mb=peak_mem, base_mem_mb=base_mem,
        num_evals=X_train_sel.shape[1],
        missing_flags=None, labels_for_leak=y_test,
        cache_hit=cache_hit,
    )
metrics.update({
    "FS_Method": FS_METHOD,
    "Classifier": "GAN",
    "NumFeatures": input_dim,
    "Energy(J)": energy,
    "Carbon(gCO2e)": carbon,
    "EDP(J*s)": edp,
    "Timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
})
results.append(metrics)
print(f"GAN: Accuracy={metrics['Accuracy']:.4f}, MacroF1={metrics['MacroF1']:.4f}")

# --- 4️⃣ SAVE SUMMARY ---
summary_path = "/kaggle/working/classification_summary_all_dl.csv"
pd.DataFrame(results).to_csv(summary_path, index=False)
print(f"\n✅ Saved ALL DL metrics to: {summary_path}")



🔹 Training MLP ...
[MLP] Epoch 01 | Train Loss: 1.4317 | Val Acc: 0.6680
[MLP] Epoch 02 | Train Loss: 0.8898 | Val Acc: 0.7375
[MLP] Epoch 03 | Train Loss: 0.7266 | Val Acc: 0.7575
[MLP] Epoch 04 | Train Loss: 0.6488 | Val Acc: 0.7440
[MLP] Epoch 05 | Train Loss: 0.5967 | Val Acc: 0.7665
[MLP] Epoch 06 | Train Loss: 0.5482 | Val Acc: 0.7695
[MLP] Epoch 07 | Train Loss: 0.5016 | Val Acc: 0.7760
[MLP] Epoch 08 | Train Loss: 0.4813 | Val Acc: 0.7885
[MLP] Epoch 09 | Train Loss: 0.4436 | Val Acc: 0.7865
[MLP] Epoch 10 | Train Loss: 0.4244 | Val Acc: 0.7660
[MLP] Epoch 11 | Train Loss: 0.3920 | Val Acc: 0.7945
[MLP] Epoch 12 | Train Loss: 0.3820 | Val Acc: 0.8005
[MLP] Epoch 13 | Train Loss: 0.3578 | Val Acc: 0.7895
[MLP] Epoch 14 | Train Loss: 0.3330 | Val Acc: 0.7990
[MLP] Epoch 15 | Train Loss: 0.3209 | Val Acc: 0.8000
[MLP] Epoch 16 | Train Loss: 0.3180 | Val Acc: 0.7980
[MLP] Epoch 17 | Train Loss: 0.2990 | Val Acc: 0.8010
[MLP] Epoch 18 | Train Loss: 0.2942 | Val Acc: 0.7975
[MLP] Ep

In [9]:
!pip install --quiet gspread gspread_dataframe oauth2client

In [10]:
import gspread
from gspread_dataframe import set_with_dataframe
from oauth2client.service_account import ServiceAccountCredentials

def connect_to_google_sheet(sheet_url, credentials_path="sheets-credentials.json"):
    scope = [
        "https://spreadsheets.google.com/feeds",
        "https://www.googleapis.com/auth/drive",
    ]
    creds = ServiceAccountCredentials.from_json_keyfile_name(credentials_path, scope)
    client = gspread.authorize(creds)
    sheet = client.open_by_url(sheet_url)
    return sheet.sheet1

def append_metrics_to_sheet(worksheet, metrics_df):
    existing = pd.DataFrame(worksheet.get_all_records())
    combined = pd.concat([existing, metrics_df], ignore_index=True)
    worksheet.clear()
    set_with_dataframe(worksheet, combined)
    print("Metrics appended successfully!")

SHEET_URL = "https://docs.google.com/spreadsheets/d/1zKdcBjfuTdeSnJ_8YAbwRv8KZdSL6EAyDOo7DClNeJg/edit?gid=0#gid=0"
CREDENTIAL_PATH = "/kaggle/input/api-json/time-complexity-474404-ba31fb86904e.json"

worksheet = connect_to_google_sheet(SHEET_URL, CREDENTIAL_PATH)
append_metrics_to_sheet(worksheet, pd.DataFrame(results))

Metrics appended successfully!
